# Interpreting relational structures in Language Models
Original Paper: https://aclanthology.org/2026.acl-long.2194/

Discourse Representation Theory formalizes how entities and relationships across entities are encoded across long-context sentences. Namely, we can decompose discourse into discourse referents (entities) and conditions (relationships between entities).

Cell-Based Representation (CBR) is a linear subspace that can encode these entities and relationships. Due to its linear design, we can also modify the representations via linear steering in CBR space.

## Preliminaries
Sentences are decomposed into entities and conditions (here, attributes) in the following way:
```json
{
  "ents": ["Sean", "Jose", "Dean"],
  // attributes: [j, j+1, j+2] where j is also the index in ents
  "atts1": ["Phoenix", "Austin", "Dallas"],
  "atts2": ["Perm", "Berlin", "Detroit"],
  "atts3": ["Kansas", "London", "Seattle"],
  "atts4": ["Miami", "Hamilton", "Paris"],
  "input": " Sean was born in Phoenix and currently lives in Perm, he loves Kansas and dislike Miami .  Jose was born in Austin and currently lives in Berlin, he loves London and dislike Hamilton .  Dean was born in Dallas and currently lives in Detroit, he loves Seattle and dislike Paris .",
  // relations
  "t_input": "| Name | Birthplace | Lived City | Loved City| Disliked City || Sean | Phoenix | Perm | Kansas | Miami || Jose | Austin | Berlin | London | Hamilton || Dean | Dallas | Detroit | Seattle | Paris |",
  "s_input": "Sean, who hails from Phoenix, currently resides in Perm. He has a fondness for Kansas but has a strong dislike for Miami. ..."
}
```
We translate these into iterable arrays in the following way
```python
ents = ["Sean", "Jose", "Dean"]
atts = [
    # Sean, Jose, Dean
  ["Phoenix", "Austin", "Dallas"],      # Birthplace
  ["Perm", "Berlin", "Detroit"],        # Lived City
  ["Kansas", "London", "Seattle"],      # Loved City
  ["Miami", "Hamilton", "Paris"],       # Disliked City
]
rel = ["Birthplace", "Lived City", "Loved City", "Disliked City"]
```


In [13]:
import json
import numpy as np

In [28]:
with open("data/CBR/city_tts_all.jsonl", "r") as file:
    entries=[]
    for l in file:
        r = json.loads(l)
        entries.append(r)
print("entries", entries[:20])

# 1. Construct Indexed Relational Scheme (IRS) Table
for entry in entries:
    ents = entry["ents"]
    atts = [entry[f"atts{i}"] for i in range(1, 5)]
    rel = [
        [el.strip() for el in row.split("|") if el.strip()]
        for row in entry["t_input"].split("||")
    ]
    rel = rel[0][1:]

    # rows = ents, cols = relations
    n, m = len(ents), len(rel)
    A = np.empty((n, m), dtype=object)
    IRS = []

    for i in range(n):  # i = entity
        for j in range(m):  # j = relation
            A[i, j] = atts[j][i]  # store attribute token
            IRS.append([i + 1, j + 1, A[i, j]])

print("A[:5]", A[:5])
print("IRS", IRS[:12])


entries [{'ents': ['Sean', 'Jose', 'Dean'], 'atts1': ['Phoenix', 'Austin', 'Dallas'], 'atts2': ['Perm', 'Berlin', 'Detroit'], 'atts3': ['Kansas', 'London', 'Seattle'], 'atts4': ['Miami', 'Hamilton', 'Paris'], 'input': ' Sean was born in Phoenix and currently lives in Perm, he loves Kansas and dislike Miami .  Jose was born in Austin and currently lives in Berlin, he loves London and dislike Hamilton .  Dean was born in Dallas and currently lives in Detroit, he loves Seattle and dislike Paris .', 'input_p_cr': ' Sean loves Kansas and was born in Phoenix, he dislikes Miami and currently lives in Perm .  Jose loves London and was born in Austin, he dislikes Hamilton and currently lives in Berlin .  Dean loves Seattle and was born in Dallas, he dislikes Paris and currently lives in Detroit .', 'input_r_cr': ' Sean works in Phoenix and studies in Perm, he travels to Kansas and invests in Miami .  Jose works in Austin and studies in Berlin, he travels to London and invests in Hamilton .  Dea